In [ ]:

# 1. INSTALL LIBRARIES
!pip install -q scikit-learn matplotlib pandas seaborn

# 2. IMPORT LIBRARIES
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import time
import warnings

from sklearn.datasets import fetch_olivetti_faces
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

warnings.filterwarnings("ignore")

np.random.seed(42)

print("Libraries imported successfully.")
# 3. LOAD OLIVETTI FACES DATASET
print("\nLoading Olivetti Faces dataset...")

data = fetch_olivetti_faces(
    shuffle=True,
    random_state=42
)

images = data.images
labels = data.target

print("Number of images :", len(images))
print("Image size       :", images.shape[1], "x", images.shape[2])
print("Number of people :", len(np.unique(labels)))

# 4. DISPLAY SAMPLE IMAGES


plt.figure(figsize=(12, 7))

for i in range(20):

    plt.subplot(4, 5, i + 1)

    plt.imshow(
        images[i],
        cmap="gray"
    )

    plt.title(
        f"Person {labels[i]}"
    )

    plt.axis("off")

plt.suptitle(
    "Sample Images from Olivetti Faces Dataset",
    fontsize=16
)

plt.tight_layout()
plt.show()

# 5. CONVERT IMAGES INTO DATA MATRIX X
# Each image:
#       64 x 64 = 4096 pixels
# X:
#       4096 x Number_of_images
# NMF representation:
#       X = W H

X = images.reshape(
    len(images),
    -1
).T

# Ensure non-negative values
X = np.maximum(X, 0)

print("\nData matrix X shape:", X.shape)

# 6. TRAIN / TEST SPLIT
indices = np.arange(
    X.shape[1]
)

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.25,
    random_state=42,
    stratify=labels
)

X_train = X[:, train_idx]
X_test = X[:, test_idx]

y_train = labels[train_idx]
y_test = labels[test_idx]

print("\nTraining images:", X_train.shape[1])
print("Testing images :", X_test.shape[1])

# 7. PARAMETERS
# NMF rank
RANK = 40

# Deep NMF ranks
RANK_LAYER_1 = 40
RANK_LAYER_2 = 20

# Maximum iterations
MAX_ITER = 300

# Regularization parameter
ALPHA = 0.05

# Prevent division by zero
EPS = 1e-10

RANDOM_STATE = 42

print("\n================ PARAMETERS ================")
print("NMF rank             :", RANK)
print("DNBMF layer 1 rank   :", RANK_LAYER_1)
print("DNBMF layer 2 rank   :", RANK_LAYER_2)
print("Maximum iterations   :", MAX_ITER)
print("Regularization alpha :", ALPHA)
print("============================================")
# 8. NORMALIZE BASIS COLUMNS
def normalize_columns(W, eps=1e-10):

    norms = np.linalg.norm(
        W,
        axis=0
    )

    norms[norms < eps] = 1.0

    return W / norms

# 9. BASIC NMF

# X = W H
class BasicNMF:

    def __init__(
        self,
        rank=40,
        max_iter=300,
        random_state=42,
        eps=1e-10
    ):

        self.rank = rank
        self.max_iter = max_iter
        self.random_state = random_state
        self.eps = eps

        self.W = None
        self.H = None

        self.errors = []


    def initialize(
        self,
        rows,
        cols
    ):

        rng = np.random.default_rng(
            self.random_state
        )

        return np.maximum(
            rng.random(
                (rows, cols)
            ),
            self.eps
        )


    def fit(self, X):

        m, n = X.shape

        # Initialize W
        self.W = self.initialize(
            m,
            self.rank
        )

        # Initialize H
        self.H = self.initialize(
            self.rank,
            n
        )

        self.errors = []

        print("\nTraining NMF")
        print("--------------------------------")

        for iteration in range(
            self.max_iter
        ):

            # --------------------------------
            # Update H
            # --------------------------------

            numerator_H = (
                self.W.T @ X
            )

            denominator_H = (
                self.W.T
                @ self.W
                @ self.H
                + self.eps
            )

            self.H *= (
                numerator_H
                /
                denominator_H
            )

            self.H = np.maximum(
                self.H,
                self.eps
            )

            # Update W

            numerator_W = (
                X @ self.H.T
            )

            denominator_W = (
                self.W
                @ self.H
                @ self.H.T
                + self.eps
            )

            self.W *= (
                numerator_W
                /
                denominator_W
            )

            self.W = np.maximum(
                self.W,
                self.eps
            )


            # Normalize basis
            self.W = normalize_columns(
                self.W
            )


            # --------------------------------
            # Reconstruction error
            # --------------------------------

            reconstruction = (
                self.W @ self.H
            )

            error = np.linalg.norm(
                X - reconstruction,
                ord="fro"
            )

            self.errors.append(
                error
            )

            if (
                iteration == 0
                or
                (iteration + 1) % 50 == 0
            ):

                print(
                    f"Iteration {iteration + 1:3d} | "
                    f"Error = {error:.4f}"
                )

        return self


    def transform(self, X):

        # Moore-Penrose pseudo-inverse
        W_pinv = np.linalg.pinv(
            self.W
        )

        features = (
            W_pinv @ X
        )

        return features.T


    def reconstruct(self, X):

        features = self.transform(
            X
        )

        reconstruction = (
            self.W
            @ features.T
        )

        return reconstruction


# 10. DNBMF
# UNDERLYING BASIS IMAGES LEARNING
# X  = W1 H1
# W1 = W2 H2
# Therefore:
# X = W2 H2 H1

class DNBMF:

    def __init__(
        self,
        rank1=40,
        rank2=20,
        max_iter=300,
        random_state=42,
        eps=1e-10
    ):

        self.rank1 = rank1
        self.rank2 = rank2
        self.max_iter = max_iter
        self.random_state = random_state
        self.eps = eps

        self.W1 = None
        self.H1 = None

        self.W2 = None
        self.H2 = None

        self.errors = []


    def initialize(
        self,
        rows,
        cols,
        seed
    ):

        rng = np.random.default_rng(
            seed
        )

        return np.maximum(
            rng.random(
                (rows, cols)
            ),
            self.eps
        )


    def fit(self, X):

        m, n = X.shape

        # FIRST LAYER

        self.W1 = self.initialize(
            m,
            self.rank1,
            self.random_state
        )

        self.H1 = self.initialize(
            self.rank1,
            n,
            self.random_state + 100
        )

        print("\nTraining DNBMF - Layer 1")
        print("--------------------------------")

        for iteration in range(
            self.max_iter
        ):

            # Update H1

            numerator_H = (
                self.W1.T @ X
            )

            denominator_H = (
                self.W1.T
                @ self.W1
                @ self.H1
                + self.eps
            )

            self.H1 *= (
                numerator_H
                /
                denominator_H
            )

            self.H1 = np.maximum(
                self.H1,
                self.eps
            )


            # Update W1

            numerator_W = (
                X @ self.H1.T
            )

            denominator_W = (
                self.W1
                @ self.H1
                @ self.H1.T
                + self.eps
            )

            self.W1 *= (
                numerator_W
                /
                denominator_W
            )

            self.W1 = np.maximum(
                self.W1,
                self.eps
            )

            self.W1 = normalize_columns(
                self.W1
            )

        # SECOND LAYER

        self.W2 = self.initialize(
            m,
            self.rank2,
            self.random_state + 1
        )

        self.H2 = self.initialize(
            self.rank2,
            self.rank1,
            self.random_state + 101
        )

        print("\nTraining DNBMF - Layer 2")
        print("--------------------------------")

        self.errors = []

        for iteration in range(
            self.max_iter
        ):

            # Update H2
            #
            # W1 ≈ W2 H2

            numerator_H2 = (
                self.W2.T
                @ self.W1
            )

            denominator_H2 = (
                self.W2.T
                @ self.W2
                @ self.H2
                + self.eps
            )

            self.H2 *= (
                numerator_H2
                /
                denominator_H2
            )

            self.H2 = np.maximum(
                self.H2,
                self.eps
            )

            # Update W2

            numerator_W2 = (
                self.W1
                @ self.H2.T
            )

            denominator_W2 = (
                self.W2
                @ self.H2
                @ self.H2.T
                + self.eps
            )

            self.W2 *= (
                numerator_W2
                /
                denominator_W2
            )

            self.W2 = np.maximum(
                self.W2,
                self.eps
            )

            self.W2 = normalize_columns(
                self.W2
            )

            # Complete reconstruction
            #
            # X ≈ W2 H2 H1

            reconstruction = (
                self.W2
                @ self.H2
                @ self.H1
            )

            error = np.linalg.norm(
                X - reconstruction,
                ord="fro"
            )

            self.errors.append(
                error
            )

            if (
                iteration == 0
                or
                (iteration + 1) % 50 == 0
            ):

                print(
                    f"Iteration {iteration + 1:3d} | "
                    f"Error = {error:.4f}"
                )

        return self


    def transform(self, X):

        # Final underlying basis
        W_pinv = np.linalg.pinv(
            self.W2
        )

        features = (
            W_pinv @ X
        )

        return features.T


    def reconstruct(self, X):

        features = self.transform(
            X
        )

        reconstruction = (
            self.W2
            @ features.T
        )

        return reconstruction

# 11. RDNBMF
# REGULARIZED DEEP NMF
# Adds regularization to the basis-image learning.

class RDNBMF:

    def __init__(
        self,
        rank1=40,
        rank2=20,
        alpha=0.05,
        max_iter=300,
        random_state=42,
        eps=1e-10
    ):

        self.rank1 = rank1
        self.rank2 = rank2
        self.alpha = alpha
        self.max_iter = max_iter
        self.random_state = random_state
        self.eps = eps

        self.W1 = None
        self.H1 = None

        self.W2 = None
        self.H2 = None

        self.errors = []


    def initialize(
        self,
        rows,
        cols,
        seed
    ):

        rng = np.random.default_rng(
            seed
        )

        return np.maximum(
            rng.random(
                (rows, cols)
            ),
            self.eps
        )


    def fit(self, X):

        m, n = X.shape

        # FIRST LAYER

        self.W1 = self.initialize(
            m,
            self.rank1,
            self.random_state
        )

        self.H1 = self.initialize(
            self.rank1,
            n,
            self.random_state + 100
        )

        print("\nTraining RDNBMF - Layer 1")
        print("--------------------------------")

        for iteration in range(
            self.max_iter
        ):

            # Update H1

            numerator_H = (
                self.W1.T @ X
            )

            denominator_H = (
                self.W1.T
                @ self.W1
                @ self.H1
                + self.eps
            )

            self.H1 *= (
                numerator_H
                /
                denominator_H
            )

            self.H1 = np.maximum(
                self.H1,
                self.eps
            )


            # Regularized W1 update

            r = self.rank1

            A = (
                np.ones(
                    (r, r)
                )
                / r
            )

            numerator_W = (
                X @ self.H1.T
                +
                self.alpha * self.W1
            )

            denominator_W = (
                self.W1
                @ self.H1
                @ self.H1.T
                +
                self.alpha
                *
                self.W1
                @ A
                +
                self.eps
            )

            self.W1 *= (
                numerator_W
                /
                denominator_W
            )

            self.W1 = np.maximum(
                self.W1,
                self.eps
            )

            self.W1 = normalize_columns(
                self.W1
            )


        # ====================================================
        # SECOND LAYER
        # ====================================================

        self.W2 = self.initialize(
            m,
            self.rank2,
            self.random_state + 1
        )

        self.H2 = self.initialize(
            self.rank2,
            self.rank1,
            self.random_state + 101
        )

        print("\nTraining RDNBMF - Layer 2")
        print("--------------------------------")

        self.errors = []

        for iteration in range(
            self.max_iter
        ):

            # Update H2

            numerator_H2 = (
                self.W2.T
                @ self.W1
            )

            denominator_H2 = (
                self.W2.T
                @ self.W2
                @ self.H2
                + self.eps
            )

            self.H2 *= (
                numerator_H2
                /
                denominator_H2
            )

            self.H2 = np.maximum(
                self.H2,
                self.eps
            )

            # Regularized W2

            r = self.rank2

            A = (
                np.ones(
                    (r, r)
                )
                / r
            )

            numerator_W2 = (
                self.W1
                @ self.H2.T
                +
                self.alpha * self.W2
            )

            denominator_W2 = (
                self.W2
                @ self.H2
                @ self.H2.T
                +
                self.alpha
                *
                self.W2
                @ A
                +
                self.eps
            )

            self.W2 *= (
                numerator_W2
                /
                denominator_W2
            )

            self.W2 = np.maximum(
                self.W2,
                self.eps
            )

            self.W2 = normalize_columns(
                self.W2
            )

            # Full reconstruction
            #
            # X ≈ W2 H2 H1

            reconstruction = (
                self.W2
                @ self.H2
                @ self.H1
            )

            error = np.linalg.norm(
                X - reconstruction,
                ord="fro"
            )

            self.errors.append(
                error
            )

            if (
                iteration == 0
                or
                (iteration + 1) % 50 == 0
            ):

                print(
                    f"Iteration {iteration + 1:3d} | "
                    f"Error = {error:.4f}"
                )

        return self


    def transform(self, X):

        W_pinv = np.linalg.pinv(
            self.W2
        )

        features = (
            W_pinv @ X
        )

        return features.T


    def reconstruct(self, X):

        features = self.transform(
            X
        )

        reconstruction = (
            self.W2
            @ features.T
        )

        return reconstruction

# 12. TRAIN NMF

print("\n")
print("=" * 60)
print("TRAINING NMF")
print("=" * 60)

start = time.time()

nmf_model = BasicNMF(
    rank=RANK,
    max_iter=MAX_ITER,
    random_state=RANDOM_STATE
)

nmf_model.fit(
    X_train
)

nmf_time = time.time() - start

print(
    f"\nNMF training time: "
    f"{nmf_time:.2f} seconds"
)


# ============================================================
# 13. TRAIN DNBMF
# ============================================================

print("\n")
print("=" * 60)
print("TRAINING DNBMF")
print("=" * 60)

start = time.time()

dnbmf_model = DNBMF(
    rank1=RANK_LAYER_1,
    rank2=RANK_LAYER_2,
    max_iter=MAX_ITER,
    random_state=RANDOM_STATE
)

dnbmf_model.fit(
    X_train
)

dnbmf_time = time.time() - start

print(
    f"\nDNBMF training time: "
    f"{dnbmf_time:.2f} seconds"
)


# ============================================================
# 14. TRAIN RDNBMF
# ============================================================

print("\n")
print("=" * 60)
print("TRAINING RDNBMF")
print("=" * 60)

start = time.time()

rdnbmf_model = RDNBMF(
    rank1=RANK_LAYER_1,
    rank2=RANK_LAYER_2,
    alpha=ALPHA,
    max_iter=MAX_ITER,
    random_state=RANDOM_STATE
)

rdnbmf_model.fit(
    X_train
)

rdnbmf_time = time.time() - start

print(
    f"\nRDNBMF training time: "
    f"{rdnbmf_time:.2f} seconds"
)

# 15. FEATURE EXTRACTION

# Feature extraction:
#
#       H = W† X
#
# where W† is Moore-Penrose pseudo-inverse.

print("\n")
print("=" * 60)
print("FEATURE EXTRACTION")
print("=" * 60)


# NMF features

nmf_train_features = (
    nmf_model.transform(
        X_train
    )
)

nmf_test_features = (
    nmf_model.transform(
        X_test
    )
)


# DNBMF features

dnbmf_train_features = (
    dnbmf_model.transform(
        X_train
    )
)

dnbmf_test_features = (
    dnbmf_model.transform(
        X_test
    )
)


# RDNBMF features

rdnbmf_train_features = (
    rdnbmf_model.transform(
        X_train
    )
)

rdnbmf_test_features = (
    rdnbmf_model.transform(
        X_test
    )
)


print(
    "NMF feature shape   :",
    nmf_train_features.shape
)

print(
    "DNBMF feature shape :",
    dnbmf_train_features.shape
)

print(
    "RDNBMF feature shape:",
    rdnbmf_train_features.shape
)

# 16. NEAREST NEIGHBOR CLASSIFIER

def evaluate_model(
    train_features,
    test_features,
    y_train,
    y_test,
    name
):

    classifier = KNeighborsClassifier(
        n_neighbors=1
    )

    classifier.fit(
        train_features,
        y_train
    )

    predictions = classifier.predict(
        test_features
    )

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    print(
        f"\n{name} Recognition Accuracy: "
        f"{accuracy * 100:.2f}%"
    )

    return (
        classifier,
        predictions,
        accuracy
    )
# 17. EVALUATE NMF
nmf_knn, nmf_pred, nmf_accuracy = (
    evaluate_model(
        nmf_train_features,
        nmf_test_features,
        y_train,
        y_test,
        "NMF"
    )
)

# 18. EVALUATE DNBMF
dnbmf_knn, dnbmf_pred, dnbmf_accuracy = (
    evaluate_model(
        dnbmf_train_features,
        dnbmf_test_features,
        y_train,
        y_test,
        "DNBMF"
    )
)

# 19. EVALUATE RDNBMF

rdnbmf_knn, rdnbmf_pred, rdnbmf_accuracy = (
    evaluate_model(
        rdnbmf_train_features,
        rdnbmf_test_features,
        y_train,
        y_test,
        "RDNBMF"
    )
)
# 20. FINAL RESULTS TABLE

results = pd.DataFrame({

    "Method": [
        "NMF",
        "DNBMF",
        "RDNBMF"
    ],

    "Accuracy (%)": [
        nmf_accuracy * 100,
        dnbmf_accuracy * 100,
        rdnbmf_accuracy * 100
    ],

    "Training Time (sec)": [
        nmf_time,
        dnbmf_time,
        rdnbmf_time
    ]
})


print("\n")
print("=" * 60)
print("FINAL IMPLEMENTATION RESULTS")
print("=" * 60)

display(
    results.round(2)
)

# 21. ACCURACY COMPARISON

plt.figure(
    figsize=(9, 6)
)

bars = plt.bar(
    results["Method"],
    results["Accuracy (%)"]
)

plt.xlabel(
    "Method"
)

plt.ylabel(
    "Recognition Accuracy (%)"
)

plt.title(
    "NMF vs DNBMF vs RDNBMF"
)

plt.ylim(
    0,
    100
)

for bar in bars:

    height = bar.get_height()

    plt.text(
        bar.get_x()
        + bar.get_width() / 2,
        height + 1,
        f"{height:.2f}%",
        ha="center",
        fontsize=11
    )

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "accuracy_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
# 22. CONVERGENCE COMPARISON

plt.figure(
    figsize=(10, 6)
)

plt.plot(
    nmf_model.errors,
    label="NMF"
)

plt.plot(
    dnbmf_model.errors,
    label="DNBMF"
)

plt.plot(
    rdnbmf_model.errors,
    label="RDNBMF"
)

plt.xlabel(
    "Iteration"
)

plt.ylabel(
    "Frobenius Reconstruction Error"
)

plt.title(
    "Convergence Comparison"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "convergence_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# 23. FUNCTION TO DISPLAY BASIS IMAGES
def show_basis_images(
    W,
    title,
    num_images=20
):

    num_images = min(
        num_images,
        W.shape[1]
    )

    plt.figure(
        figsize=(12, 8)
    )

    for i in range(num_images):

        plt.subplot(
            4,
            5,
            i + 1
        )

        basis = W[:, i].reshape(
            64,
            64
        )

        plt.imshow(
            basis,
            cmap="gray"
        )

        plt.title(
            f"Basis {i + 1}"
        )

        plt.axis("off")

    plt.suptitle(
        title,
        fontsize=16
    )

    plt.tight_layout()

    plt.show()


# 24. NMF BASIS IMAGES
show_basis_images(
    nmf_model.W,
    "NMF Learned Basis Images"
)

# 25. DNBMF LAYER 1 BASIS
show_basis_images(
    dnbmf_model.W1,
    "DNBMF - Layer 1 Basis Images"
)

# 26. DNBMF FINAL BASIS
show_basis_images(
    dnbmf_model.W2,
    "DNBMF - Layer 2 Underlying Basis Images"
)

# 27. RDNBMF LAYER 1 BASIS
show_basis_images(
    rdnbmf_model.W1,
    "RDNBMF - Layer 1 Basis Images"
)

# 28. RDNBMF FINAL BASIS
show_basis_images(
    rdnbmf_model.W2,
    "RDNBMF - Final Underlying Basis Images"
)

# 29. ORIGINAL VS RECONSTRUCTED FACE


sample = 0

original = X_test[
    :,
    sample
]


nmf_reconstructed = (
    nmf_model.reconstruct(
        X_test[:, sample:sample + 1]
    )[:, 0]
)


dnbmf_reconstructed = (
    dnbmf_model.reconstruct(
        X_test[:, sample:sample + 1]
    )[:, 0]
)


rdnbmf_reconstructed = (
    rdnbmf_model.reconstruct(
        X_test[:, sample:sample + 1]
    )[:, 0]
)


plt.figure(
    figsize=(14, 4)
)


plt.subplot(
    1,
    4,
    1
)

plt.imshow(
    original.reshape(64, 64),
    cmap="gray"
)

plt.title(
    "Original"
)

plt.axis("off")


plt.subplot(
    1,
    4,
    2
)

plt.imshow(
    nmf_reconstructed.reshape(64, 64),
    cmap="gray"
)

plt.title(
    "NMF"
)

plt.axis("off")


plt.subplot(
    1,
    4,
    3
)

plt.imshow(
    dnbmf_reconstructed.reshape(64, 64),
    cmap="gray"
)

plt.title(
    "DNBMF"
)

plt.axis("off")


plt.subplot(
    1,
    4,
    4
)

plt.imshow(
    rdnbmf_reconstructed.reshape(64, 64),
    cmap="gray"
)

plt.title(
    "RDNBMF"
)

plt.axis("off")


plt.suptitle(
    "Original vs Reconstructed Face",
    fontsize=16
)

plt.tight_layout()

plt.savefig(
    "reconstruction_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# 30. ACTUAL VS PREDICTED PERSON

plt.figure(
    figsize=(12, 4)
)


plt.subplot(
    1,
    3,
    1
)

plt.imshow(
    X_test[:, sample].reshape(64, 64),
    cmap="gray"
)

plt.title(
    f"NMF\n"
    f"Actual: Person {y_test[sample]}\n"
    f"Predicted: Person {nmf_pred[sample]}"
)

plt.axis("off")


plt.subplot(
    1,
    3,
    2
)

plt.imshow(
    X_test[:, sample].reshape(64, 64),
    cmap="gray"
)

plt.title(
    f"DNBMF\n"
    f"Actual: Person {y_test[sample]}\n"
    f"Predicted: Person {dnbmf_pred[sample]}"
)

plt.axis("off")


plt.subplot(
    1,
    3,
    3
)

plt.imshow(
    X_test[:, sample].reshape(64, 64),
    cmap="gray"
)

plt.title(
    f"RDNBMF\n"
    f"Actual: Person {y_test[sample]}\n"
    f"Predicted: Person {rdnbmf_pred[sample]}"
)

plt.axis("off")


plt.tight_layout()

plt.show()

# 31. CONFUSION MATRIX - NMF

cm_nmf = confusion_matrix(
    y_test,
    nmf_pred
)

plt.figure(
    figsize=(9, 7)
)

sns.heatmap(
    cm_nmf,
    cmap="Blues",
    cbar=True
)

plt.title(
    "NMF Confusion Matrix"
)

plt.xlabel(
    "Predicted Person"
)

plt.ylabel(
    "Actual Person"
)

plt.tight_layout()

plt.show()
# 32. CONFUSION MATRIX - DNBMF

cm_dnbmf = confusion_matrix(
    y_test,
    dnbmf_pred
)

plt.figure(
    figsize=(9, 7)
)

sns.heatmap(
    cm_dnbmf,
    cmap="Blues",
    cbar=True
)

plt.title(
    "DNBMF Confusion Matrix"
)

plt.xlabel(
    "Predicted Person"
)

plt.ylabel(
    "Actual Person"
)

plt.tight_layout()

plt.show()

# 33. CONFUSION MATRIX - RDNBMF
cm_rdnbmf = confusion_matrix(
    y_test,
    rdnbmf_pred
)

plt.figure(
    figsize=(9, 7)
)

sns.heatmap(
    cm_rdnbmf,
    cmap="Blues",
    cbar=True
)

plt.title(
    "RDNBMF Confusion Matrix"
)

plt.xlabel(
    "Predicted Person"
)

plt.ylabel(
    "Actual Person"
)

plt.tight_layout()

plt.show()

# 34. CLASSIFICATION REPORT


print("\n")
print("=" * 60)
print("NMF CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        y_test,
        nmf_pred
    )
)


print("\n")
print("=" * 60)
print("DNBMF CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        y_test,
        dnbmf_pred
    )
)


print("\n")
print("=" * 60)
print("RDNBMF CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        y_test,
        rdnbmf_pred
    )
)

# 35. FINAL COMPARISON TABLE

final_comparison = pd.DataFrame({

    "Method": [
        "NMF",
        "DNBMF",
        "RDNBMF"
    ],

    "Architecture": [
        "Shallow NMF",
        "Deep UBIL",
        "Deep UBIL + Regularization"
    ],

    "Final Basis Rank": [
        RANK,
        RANK_LAYER_2,
        RANK_LAYER_2
    ],

    "Accuracy (%)": [
        nmf_accuracy * 100,
        dnbmf_accuracy * 100,
        rdnbmf_accuracy * 100
    ],

    "Training Time (sec)": [
        nmf_time,
        dnbmf_time,
        rdnbmf_time
    ]
})


print("\n")
print("=" * 70)
print("FINAL METHOD COMPARISON")
print("=" * 70)

display(
    final_comparison.round(2)
)

# 36. FIND BEST METHOD

best_index = (
    final_comparison[
        "Accuracy (%)"
    ].idxmax()
)

best_method = (
    final_comparison.loc[
        best_index,
        "Method"
    ]
)

best_accuracy = (
    final_comparison.loc[
        best_index,
        "Accuracy (%)"
    ]
)

print("\n")
print("=" * 60)
print("BEST PERFORMING METHOD")
print("=" * 60)

print(
    f"Best Method   : {best_method}"
)

print(
    f"Accuracy      : {best_accuracy:.2f}%"
)


# 37. SAVE RESULTS
results.to_csv(
    "implementation_results.csv",
    index=False
)

final_comparison.to_csv(
    "final_method_comparison.csv",
    index=False
)


# 38. SAVE RECONSTRUCTION ERROR

error_table = pd.DataFrame({

    "Iteration": np.arange(
        1,
        MAX_ITER + 1
    ),

    "NMF Error": nmf_model.errors,

    "DNBMF Error": dnbmf_model.errors,

    "RDNBMF Error": rdnbmf_model.errors

})

error_table.to_csv(
    "convergence_errors.csv",
    index=False
)

# 39. FINAL PROJECT SUMMARY


print("\n")
print("=" * 70)
print("                  PROJECT COMPLETED")
print("=" * 70)

print(
    f"\nNMF Accuracy    : "
    f"{nmf_accuracy * 100:.2f}%"
)

print(
    f"DNBMF Accuracy  : "
    f"{dnbmf_accuracy * 100:.2f}%"
)

print(
    f"RDNBMF Accuracy : "
    f"{rdnbmf_accuracy * 100:.2f}%"
)

print("\nArchitecture:")

print(
    "NMF    : X = W H"
)

print(
    "DNBMF  : X = W2 H2 H1"
)

print(
    "RDNBMF : X = W2 H2 H1 + regularization"
)

print(
    "\nFeature extraction:"
)

print(
    "H = W† X"
)

print(
    "\nClassifier:"
)

print(
    "1-Nearest Neighbor"
)

print("\nSaved files:")

print(
    "1. accuracy_comparison.png"
)

print(
    "2. convergence_comparison.png"
)

print(
    "3. reconstruction_comparison.png"
)

print(
    "4. implementation_results.csv"
)

print(
    "5. final_method_comparison.csv"
)

print(
    "6. convergence_errors.csv"
)

print("\n")
print("=" * 70)
print("                    DONE")
print("=" * 70)